In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## Step 1: സ്ട്രക്ചർഡ് ഔട്ട്പുട്ടുകൾക്കായുള്ള പൈഡാന്റിക് മോഡലുകൾ സംവരിക്കുക

ഈ മോഡലുകൾ ഏജന്റുകൾ തിരികെ നൽകുന്ന **സ്കീമ** നിർവചിക്കുന്നു. Pydantic ഉപയോഗിച്ച് `response_format` ഉപയോഗിക്കുന്നത് ഉറപ്പാക്കുന്നു:
- ✅ ടൈപ്പ്-സേഫ് ഡേറ്റാ എക്സ്ട്രാക്ഷൻ
- ✅ സ്വയംപരിശോധന
- ✅ സ്വതന്ത്ര-വാചക പ്രതികരണങ്ങളിൽ നിന്ന് പാർസിംഗ് എററുകൾ ഇല്ല
- ✅ ഫീൽഡുകളെ അടിസ്ഥാനമാക്കി എളുപ്പത്തിലുള്ള വ്യത്യസ്ത മാർഗനിർദ്ദേശം


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Step 2: ഹോട്ടൽ ബുക്കിംഗ് ടൂൾ സൃഷ്ടിക്കുക

റൂമുകൾ ലഭ്യമായുണ്ടോ എന്ന് പരിശോധിക്കാൻ **availability_agent** വിളിക്കുന്ന ടൂളാണ് ഇത്. `@ai_function` ഡെക്കറേറ്റർ ഉപയോഗിക്കുന്നത്:
- ഒരു പൈത്തൺ ഫംഗ്ഷനെ AI-കോളബിൾ ടൂളായി മാറ്റാൻ
- LLM-க்கு JSON സ്കീമ സ്വയം സൃഷ്ടിക്കാൻ
- പാരാമീറ്റർ സാധുത പരിശോധന കൈകാര്യം ചെയ്യാൻ
- ഏജന്റുകൾക്ക് സ്വയം ടൂൾ കോൾ ചെയ്യാൻ സാധ്യം ആക്കാൻ

ഈ ഡെമോയ്ക്ക്:
- **സ്റ്റോക്ക്‌ഹോം, സീട്ടിൽ, ടോക്യോ, ലണ്ടൻ, ആംസ്റ്റർഡാം** → റൂമുകൾ ലഭ്യമാണ് ✅
- **മറ്റുള്ള എല്ലാ നഗരങ്ങളും** → റൂമുകൾ ഇല്ല ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## റ്റിപ്പ് 3: റൂട്ടിംഗിനായി കൺഡിഷൻ ഫങ്ഷനുകൾ നിർവചിക്കുക

ഈ ഫങ്ഷനുകൾ ഏജന്റിന്റെ പ്രതികരണം പരിശോധിച്ച് പ്രവൃത്തി പ്രവാഹത്തിൽ എടുക്കേണ്ട പാത നിർണ്ണയിക്കുന്നു.

**പ്രധാന പാറ്റേൺ:**
1. സന്ദേശം `AgentExecutorResponse` ആണെന്ന് പരിശോധിക്കുക
2. ഘടനാപരమైన ഔട്ട്പുട്ട് (Pydantic മോഡൽ) പാഴ്സ് ചെയ്യുക
3. റൂട്ടിംഗ നിയന്ത്രിക്കാൻ `True` അല്ലെങ്കിൽ `False` മടങ്ങുക

വർക്ക്ഫ്ലോ ഈ കൺഡിഷനുകൾ **എഡ്ജുകളിൽ** വിലയിരുത്തുകയും അടുത്തുള്ള എക്സിക്യൂട്ടറെ ആഹ്വാനം ചെയ്യേണ്ടതെന്ന് തീരുമാനിക്കുകയും ചെയ്യും.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## പടി 4: കസ്റ്റം ഡിസ്പ്ലേ എക്സിക്യൂട്ടർ സൃഷ്ടിക്കുക

എക്സിക്യൂട്ടറുകൾ ട്രാൻസ്ഫർമേഷനുകൾ അല്ലെങ്കിൽ സൈഡ് എഫക്ടുകൾ നടത്തുകയും ചെയ്യുന്ന വർക്ക്‌ഫ്ലോ ഘടകങ്ങളാണ്. ഫൈനൽ ഫലം പ്രദർശിപ്പിക്കുന്ന ഒരു കസ്റ്റം എക്സിക്യൂട്ടർ സൃഷ്ടിക്കാനായി `@executor` ഡെക്കറേറ്റർ，我们 ഉപയോഗിക്കുന്നു.

**പ്രധാന സങ്കല്പങ്ങൾ:**
- `@executor(id="...")` - ഫംഗ്ഷനെ ഒരു വർക്ക്‌ഫ്ലോ എക്സിക്യൂട്ടറായി രജിസ്റ്റർ ചെയ്യും
- `WorkflowContext[Never, str]` - ഇൻപുട്ട്/ഔട്ട്‌പുട്ടിനുള്ള ടൈപ്പ് ഹിന്റുകൾ
- `ctx.yield_output(...)` - അന്തിമ വർക്ക്‌ഫ്ലോ ഫലം യീൽഡ് ചെയ്യുന്നു


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## ഘട്ടം 5: പരിസ്ഥിതി ചരങ്ങൾ ലോഡ് ചെയ്യുക

LLM ക്ലയന്റ് ക്രമീകരിക്കുക. ഈ ഉദാഹരണം പ്രവർത്തിക്കുന്നു:
- **GitHub മോഡലുകൾ** (GitHub ടോക്കൺ ഉപയോഗിച്ച് സൗജന്യ റീതി)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ഘട്ടം 6: ഘടനാപരമായ ഔട്ട്പുട്ടുകളുള്ള AI ഏജൻറുകൾ സൃഷ്ടിക്കുക

നാം **മൂന്നു പ്രത്യേക ഏജൻറുകളെ** സൃഷ്ടിക്കുന്നു, ഓരോന്നും ഒരു `AgentExecutor`-ൽ അവസാനിപ്പിച്ചിരിക്കുന്നു:

1. **availability_agent** - ടൂൾ ഉപയോഗിച്ച് ഹോട്ടൽ ലഭ്യത പരിശോധിക്കുന്നു  
2. **alternative_agent** - മറ്റു നഗരങ്ങൾ നിർദേശം നൽകുന്നു (റൂമുകൾ ലഭ്യമല്ലെങ്കിൽ)  
3. **booking_agent** - ബുക്കിംഗ് പ്രോത്സാഹിപ്പിക്കുന്നു (റൂമുകൾ ലഭ്യമുണ്ടെങ്കിൽ)  

**പ്രധാന സവിശേഷതകൾ:**  
- `tools=[hotel_booking]` - ഏജൻറിന് ടൂൾ നൽകുന്നു  
- `response_format=PydanticModel` - ഘടനാപരമായ JSON ഔട്ട്പുട്ട് നിർബന്ധിപ്പിക്കുന്നു  
- `AgentExecutor(..., id="...")` - പ്രവൃത്തി ഫ്ലോ ഉപയോഗത്തിനായി ഏജൻറിനെ തൊടുന്ന പാക്കേജ്


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 7: കണ്ടീഷണൽ എജുകൾ ഉപയോഗിച്ച് വർക്ക്‌ഫ്ലോ നിർമ്മിക്കുക

ഇപ്പോൾ `WorkflowBuilder` ഉപയോഗിച്ച് കണ്ടീഷണൽ റൂട്ടിംഗ് ഉപയോഗിച്ച് ഗ്രാഫ് നിർമ്മിക്കുന്നു:

**Workflow ഘടന:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**പ്രധാന മേധഡുകൾ:**
- `.set_start_executor(...)` - പ്രവേശന ബിന്ദു സജ്ജമാക്കുന്നു
- `.add_edge(from, to, condition=...)` - കണ്ടീഷണൽ എജ് ചേർക്കുന്നു
- `.build()` - വർക്ക്‌ഫ്ലോ പൂർത്തിയാക്കുന്നു


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 8: ടെസ്റ്റ് കേസ് 1 ഓടിക്കുക - ലഭ്യതയില്ലാത്ത നഗരം (പാരിസ്)

പാരിസിലെ ഹോട്ടലുകൾ അഭ്യർത്ഥിച്ച് **ലഭ്യതയില്ലാത്ത** പാത പരീക്ഷിക്കാം (നമ്മുടെ സിമുലേഷനിൽ അവിടെ മുറികളില്ല).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## ഘട്ടം 9: ടെസ്റ്റ് കേസ് 2 നടത്തുക - ലഭ്യതയുള്ള നഗരം (സ്റ്റോക്ക്ഹോം)

ഇപ്പോൾ നമ്മുടെ അനുകരണത്തിലുള്ള മുറികളുള്ള സ്റ്റോക്ക്ഹോം നഗരത്തിൽ ഹോട്ടലുകൾ അഭ്യർത്ഥിച്ച് **ലഭ്യത** പാതയെ പരിശോധിക്കാം.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## പ്രധാനപ്പെട്ട കാര്യങ്ങളും അടുത്തുപടികളും

### ✅ നിങ്ങൾ പഠിച്ചതെന്ത്:

1. **WorkflowBuilder പാറ്റേൺ**
   - പ്രവേശനപോയിന്റ് നിർവചിക്കാൻ `.set_start_executor()` ഉപയോഗിക്കുക
   - നിബന്ധനാനുസരിച്ചുള്ള മാർഗ്ഗനിർദ്ദേശത്തിന് `.add_edge(from, to, condition=...)` ഉപയോഗിക്കുക
   - വർക്ക്‌ഫ്ലോ ഫൈനലാക്കാൻ `.build()` വിളിക്കുക

2. **നിബന്ധനാനുസരിച്ചുള്ള മാർഗ്ഗനിർദ്ദേശം**
   - വീഥികാര്യങ്ങൾ `AgentExecutorResponse` പരിശോധിക്കുന്നു
   - വിന്യാസപ്പെട്ട ഔട്ട്പുട്ടുകൾ പരാമർശിച്ച് മാർഗ്ഗനിർദ്ദേശം തീരുമാനിക്കുന്നു
   - എജ്ജ് ആക്ടിവേറ്റ് ചെയ്യാൻ `True` റിട്ടേൺ ചെയ്യുക, ഒഴിവാക്കാൻ `False`

3. **ടൂൾ സംയോജനം**
   - Python ഫംഗ്ഷനുകൾ AI ടൂളുകളായി മാറ്റാൻ `@ai_function` ഉപയോഗിക്കുക
   - ഏജന്റുകൾ ആവശ്യമായപ്പോൾ ടൂളുകൾ സ്വയം വിളിക്കും
   - ടൂളുകൾ ഏജന്റുകൾക്കായി JSON റിട്ടേൺ ചെയ്യുന്നു

4. **വിന്യാസപ്പെട്ട ഔട്ട്പുട്ടുകൾ**
   - സുരക്ഷിതമായ ഡാറ്റാ എക്സ്ട്രാക്ഷനായി Pydantic മോഡലുകൾ ഉപയോഗിക്കുക
   - ഏജന്റുകൾ സൃഷ്ടിക്കുമ്പോൾ `response_format=MyModel` സജ്ജമാക്കുക
   - പ്രതികരണങ്ങൾ `Model.model_validate_json()` ഉപയോഗിച്ച് പാഴ്സ് ചെയ്യുക

5. **കസ്റ്റം എക്സിക്യൂട്ടർമാർ**
   - വർക്ക്‌ഫ്ലോ ഘടകങ്ങൾ സൃഷ്ടിക്കാൻ `@executor(id="...")` ഉപയോഗിക്കുക
   - എക്സിക്യൂട്ടർമാർ ഡാറ്റ മാറ്റുകയും ഓപ്പറേഷനുകൾ നടത്തുകയും ചെയ്യാം
   - വർക്ക്‌ഫ്ലോ ഫലം ഉത്പാദിപ്പിക്കാൻ `ctx.yield_output()` ഉപയോഗിക്കുക

### 🚀 യാഥാർത്ഥ്യ പ്രയോഗങ്ങൾ:

- **യാത്ര ബുക്കിംഗ്**: ലഭ്യത പരിശോധിക്കുക, പകരം നിർദ്ദേശിക്കുക, ഓപ്ഷനുകൾ താരതമ്യം ചെയ്യുക
- **കസ്റ്റമർ സർവീസ്**: പ്രശ്ന തരം, മനോഭാവം, മുൻഗണന അടിസ്ഥാനമാക്കി മാർഗ്ഗനിർദ്ദേശം
- **ഇ-കൊമേഴ്സ്**: ഇൻവെന്ററി പരിശോധിക്കൽ, പകരം നിർദ്ദേശിക്കൽ, ഓർഡറുകൾ കൈകാര്യം ചെയ്യൽ
- **കണ്ടന്റ് മോണിറ്ററിംഗ്**: വിഷവശ്യ സൂചികകൾ, ഉപയോഗക്കാരന്റെ ഫ്ലാഗുകൾ അടിസ്ഥാനമാക്കി മാർഗ്ഗനിർദ്ദേശം
- **അനുമതി വർക്ക്‌ഫ്ലോകൾ**: തുക, ഉപയോക്തൃ റോളുകൾ, റിസ്ക് ലെവൽ അടിസ്ഥാനമാക്കി മാർഗ്ഗനിർദ്ദേശം
- **മൾട്ടി-സ്റ്റേജ് പ്രോസസ്സ്**: ഡാറ്റാ ഗുണമേന്മ, പൂർണ്ണത എന്നിവാാടെ മാർഗ്ഗനിർദ്ദേശം

### 📚 അടുത്തുവടുക്കുന്നത്:

- കൂടുതല് സങ്കീര്‍ണ്ണ നിബന്ധനകൾ ചേർക്കുക (മൾട്ടി ക്രൈറ്റീരിയ)
- വർക്ക്‌ഫ്ലോ സ്റ്റേറ്റ് മാനേജ്മെന്റോടെ ലൂപ്പുകൾ നടപ്പിലാക്കുക
- പുനരുപയോഗയോഗ്യ ഘടകങ്ങൾക്ക് സബ്-വർക്ക്‌ഫ്ലോകൾ ചേർക്കുക
- യഥാർത്ഥ API കളുമായി സംയോജിപ്പിക്കുക (ഹോട്ടൽ ബുക്കിംഗ്, ഇൻവെന്ററി സിസ്റ്റങ്ങൾ)
- പിശകുകൾ കൈകാര്യം ചെയ്യലും ഫാല്ബാക്ക് പാതകളും ചേർക്കുക
- നിർമാണത്തിലെ വിസ്വലൈസേഷൻ ടൂളുകൾ ഉപയോഗിച്ച് വർക്ക്‌ഫ്ലോകൾ ദൃശ്യമാക്കുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
